# Gold Orders Summary Aggregation

Aggregates `product.bronze.orders` into `product.gold.orders_summary`, grouping by order date, product, and price to compute the number of orders, total products sold, and total amount per day per product.

In [0]:
%sql
-- Create the gold orders_summary table
CREATE TABLE IF NOT EXISTS product.gold.orders_summary (
    order_date           DATE     NOT NULL,
    no_of_orders         INT      NOT NULL,
    product_id           INT      NOT NULL,
    product_name         STRING   NOT NULL,
    total_products_sold  INT      NOT NULL,
    product_price        INT      NOT NULL,
    total_amount         INT      NOT NULL
)
USING DELTA;

-- Aggregate bronze.orders: group by order_date, product_id, product_name, product_price
INSERT INTO product.gold.orders_summary
SELECT
    order_date,
    COUNT(*)                 AS no_of_orders,
    product_id,
    product_name,
    SUM(product_quantity)    AS total_products_sold,
    product_price,
    SUM(product_quantity) * product_price AS total_amount
FROM product.bronze.orders
GROUP BY order_date, product_id, product_name, product_price
ORDER BY order_date, product_id;

num_affected_rows,num_inserted_rows
197,197


In [0]:
%sql
-- Verify the aggregated data
SELECT * FROM product.gold.orders_summary
ORDER BY order_date, product_id;

order_date,no_of_orders,product_id,product_name,total_products_sold,product_price,total_amount
2026-04-01,1,1,tea,1,10,10
2026-04-01,1,2,coffee,1,10,10
2026-04-02,1,1,tea,1,10,10
2026-04-02,2,2,coffee,2,10,20
2026-04-03,1,1,tea,1,10,10
2026-04-03,3,3,soap,3,10,30
2026-04-04,3,2,coffee,3,10,30
2026-04-04,1,4,paste,1,10,10
2026-04-04,2,5,cream,2,10,20
2026-04-06,3,1,tea,3,10,30
